# B1.11 · Remediation engineering

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.10 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B1.10.html)**.

| | |
|---|---|
| Open-source tooling | Semgrep OSS, pytest |
| Open-weight models | GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

A patch that passes the tests and changes the behaviour is not a fix, it is a second incident with a pull request attached. Remediation is the stage where the pipeline stops finding things and starts touching them.

## 2 · The framework

```
   patch                    what has to be true
   +----------------+       +-----------------------------+
   | fixes the bug  |  and  | behaviour unchanged         |
   |                |       | tests still pass            |
   |                |       | reviewer can follow the why |
   +----------------+       +-----------------------------+

   a patch that passes the tests and changes the behaviour is
   a second incident with a pull request attached
```

**Stage 14 — Remediation engineering.** Generate the fix, then prove it.

A model that finds bugs is useful. A model that fixes them is only useful if you
can tell a real fix from a plausible one, and plausible is exactly what language
models are optimised to produce.

There are three ways to make a finding stop firing:

1. **Fix the vulnerability** — behaviour preserved, bug gone.
2. **Remove the code** — finding gone, so is the feature.
3. **Evade the detector** — rewrite until the pattern misses.

All three make the scanner green, and an autonomous loop optimising for a green
scan will find options 2 and 3 on its own because they are cheaper.

The pipeline has an advantage a static workflow does not: Phase 4 already built
a working exploit. So the acceptance test is not "does the scanner still fire?"
It is **"does the exploit still work against the patched build?"** — which is
the only question that cannot be gamed by editing the code around the detector.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

> **About the model in this notebook.** It runs offline against a deterministic
> stand-in so the lesson executes on a Kaggle kernel with no network. The
> stand-in is not a language model and is labelled as such wherever it appears.
> To run the identical stage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 3 · The confirmed finding, with its working exploit

In [ ]:
import re, sqlite3

VULNERABLE = '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = '" + name + "'").fetchall()
'''

def build_db():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users(id INTEGER, name TEXT)")
    conn.executemany("INSERT INTO users VALUES (?,?)",
                     [(1,"dana"),(2,"sam"),(3,"o'brien")])
    return conn

def load(src):
    ns = {}; exec(compile(src, "<patch>", "exec"), ns); return ns["get_user"]

BEHAVIOUR = [("dana",[(1,"dana")]), ("sam",[(2,"sam")]),
             ("nobody",[]), ("o'brien",[(3,"o'brien")])]

def behaviour_ok(fn):
    conn = build_db(); rows = []
    for name, expected in BEHAVIOUR:
        try: got = fn(conn, name)
        except Exception as e: rows.append((name, f"raised {type(e).__name__}", False)); continue
        rows.append((name, got, got == expected))
    return rows

def exploit_works(fn):
    """The stage-12 probe, reused as the acceptance test."""
    conn = build_db()
    try: rows = fn(conn, "x' OR '1'='1")
    except Exception: return False, "probe raised — not exploitable this way"
    return len(rows) > 1, f"probe returned {len(rows)} rows"

def scanner_fires(src):
    return bool(re.search(r"execute\(\s*[\"\'][^\"\']*[\"\']\s*\+", src))

fn = load(VULNERABLE)
print("behaviour of the vulnerable build:")
for name, got, ok in behaviour_ok(fn):
    print(f"   get_user({name!r:10s}) → {str(got):18s} {'ok' if ok else 'FAILS'}")
ex, why = exploit_works(fn)
print(f"\nexploit works: {ex} — {why}")
print(f"scanner fires: {scanner_fires(VULNERABLE)}")

## 4 · Four candidate patches, three of which make CI green

In [ ]:
CANDIDATES = {
 "A · parameterise (the real fix)": '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = ?", (name,)).fetchall()
''',
 "B · delete the feature": '''
def get_user(conn, name):
    return []
''',
 "C · evade the scanner": '''
def get_user(conn, name):
    q = "SELECT id, name FROM users WHERE name = '%s'" % name
    return conn.execute(q).fetchall()
''',
 "D · escape by hand": '''
def get_user(conn, name):
    safe = name.replace("'", "''")
    return conn.execute("SELECT id, name FROM users WHERE name = '" + safe + "'").fetchall()
''',
}
print(f"{'candidate':34s}{'scanner green':>15}")
print("-" * 50)
for name, src in CANDIDATES.items():
    print(f"{name:34s}{str(not scanner_fires(src)):>15}")
print("\nThree of four are green. Only one of those is a fix.")

## 5 · The control — validate on three axes, exploit first

In [ ]:
def validate(src):
    fn = load(src)
    green = not scanner_fires(src)
    beh = behaviour_ok(fn)
    preserved = all(ok for _, _, ok in beh)
    still_exploitable, _ = exploit_works(fn)
    reasons = []
    if not green:            reasons.append("scanner still fires")
    if not preserved:        reasons.append("behaviour changed")
    if still_exploitable:    reasons.append("STILL EXPLOITABLE (stage-12 probe passes)")
    return (not reasons), green, preserved, still_exploitable, reasons

print(f"{'candidate':34s}{'scan':6s}{'behaviour':11s}{'exploitable':13s}verdict")
print("-" * 84)
accepted = []
for name, src in CANDIDATES.items():
    ok, g, b, x, reasons = validate(src)
    if ok: accepted.append(name)
    print(f"{name:34s}{str(g):6s}{str(b):11s}{str(x):13s}"
          f"{'ACCEPT' if ok else 'REJECT — ' + ', '.join(reasons)}")
print(f"\naccepted: {accepted}")
assert "A · parameterise (the real fix)" in accepted
assert "B · delete the feature" not in accepted
assert "C · evade the scanner" not in accepted

In [ ]:
# The proof-of-fix clause: the exploit must fail on the new build and
# succeed on the old one. Without both halves, "fixed" is a claim.
def proof_of_fix(old_src, new_src):
    old_ex, _ = exploit_works(load(old_src))
    new_ex, _ = exploit_works(load(new_src))
    return (old_ex and not new_ex), f"exploit on old={old_ex}, on new={new_ex}"

for name in accepted:
    ok, detail = proof_of_fix(VULNERABLE, CANDIDATES[name])
    print(f"{name:34s} proof of fix: {ok}  ({detail})")

print("\nCandidate D passes every automated check and is still the wrong answer:")
print("it reimplements the driver's escaping and will be wrong for the next")
print("input class or the next database. Nothing except a rule about MECHANISM")
print("catches that — which is the part of remediation that does not automate.")

## What you just proved

The vulnerable build passes all four behaviour cases and the exploit returns 3 rows. Candidates A, B and D make the scanner green. Validation rejects B for changed behaviour and C for remaining exploitable, accepting A and D. Proof of fix holds for both accepted patches — the exploit works on the old build and fails on the new.

## Your turn

Candidate D passes every automated gate and is still wrong. Write the rule that rejects it. You will find it has to be about which *mechanism* is acceptable, not about outcomes — and that rule belongs in your secure coding standard, not in the pipeline.

---

**Next → [B1.12 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*